# 🎨 face2sketch — Phase 2: Finetune → Caricature

**Load Phase 1 G weights, finetune on TwitterPicasso (184 pairs) for caricature style.**

| Step | Time |
|---|---|
| Download data + checkpoint | ~1 min |
| Training (100 epochs) | ~45 min T4 |
| Evaluate | ~1 min |

### What's Different from Phase 1
- **Data:** TwitterPicasso caricatures (184 pairs) — bold, exaggerated style
- **G init:** Loads Phase 1 pretrained weights (photo→drawing knowledge transfers)
- **D init:** Fresh random initialization (new style needs new discriminator)
- **LR:** 5e-5 (10× lower than Phase 1)
- **λ_L1:** 50 (half — lean more on adversarial loss for style transfer)
- **Augmentation:** More aggressive (30° rotation + affine transforms)

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/face2sketch.git /content/face2sketch 2>/dev/null
%cd /content/face2sketch
!git pull origin main

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples outputs

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print('\n✅  Setup complete.')


In [ ]:
# @title 2. Download Data + Phase1 Checkpoint from Drive (~2 min)
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/face2sketch'

# Phase 1 data (dataset/ for normalization + test/ for eval)
!cp {DRIVE}/data.zip /content/face2sketch/ 2>/dev/null
!unzip -o data.zip

# Phase 2 finetune data
!cp {DRIVE}/finetune_data.zip /content/face2sketch/ 2>/dev/null
!unzip -o finetune_data.zip

# Phase 1 checkpoint
!cp {DRIVE}/checkpoints/pix2pix_best.pt checkpoints/ 2>/dev/null

# Verify
import os
ft_photos = len(os.listdir('data/finetune/photos')) if os.path.isdir('data/finetune/photos') else 0
ft_sketch = len(os.listdir('data/finetune/sketches')) if os.path.isdir('data/finetune/sketches') else 0
ckpt_ok = os.path.exists('checkpoints/pix2pix_best.pt')

print(f"Finetune pairs: {min(ft_photos, ft_sketch)}")
print(f"Phase 1 ckpt:  {'✅' if ckpt_ok else '❌ NOT FOUND'}")

if not ckpt_ok:
    print("\nUpload pix2pix_best.pt to: MyDrive/face2sketch/checkpoints/")
if ft_photos == 0:
    print("Upload finetune_data.zip to: MyDrive/face2sketch/")
    print('Zip contents: data/finetune/photos/*.jpg + data/finetune/sketches/*.jpg')


In [ ]:
# @title 3. Finetune — 100 epochs (~45 min T4)
assert torch.cuda.is_available(), "⚠️  Enable GPU: Runtime → Change runtime type → T4 GPU"
assert os.path.exists('checkpoints/pix2pix_best.pt'), "❌ Missing Phase1 checkpoint!"
assert os.path.isdir('data/finetune'), "❌ Missing finetune data!"

%cd /content/face2sketch

print("🎯  Finetuning Phase 1 G on TwitterPicasso caricatures")
print("    G: pretrained (photo→drawing)  |  D: fresh (new style)")
print(f"    Dataset: 184 pairs  |  Epochs: 100  |  λ_L1: 50\n")

!python src/train.py --mode finetune \
    --finetune checkpoints/pix2pix_best.pt \
    --device cuda \
    --name phase2_

print("\n✅  Finetuning complete → checkpoints/phase2_best.pt")


In [ ]:
# @title 4. Evaluate — Phase 2 Gate Check
%cd /content/face2sketch
import os, glob

ckpt = 'checkpoints/phase2_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/phase2_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No Phase 2 checkpoint found.')
elif not os.path.exists('data/test/photos'):
    print('⚠️  No test data found.')
else:
    # Quick visual check
    !python src/evaluate.py --checkpoint {ckpt} --mode quick --device cuda

    # Generate comparison: Phase 1 vs Phase 2 on same photos
    print('\n📊  Comparing Phase 1 vs Phase 2 outputs...')
    p1_ckpt = 'checkpoints/pix2pix_best.pt'
    if os.path.exists(p1_ckpt):
        !python src/sample.py --checkpoint {p1_ckpt} --input-dir data/test/photos \
            --output-dir outputs/phase1_test --device cuda --n 8
    !python src/sample.py --checkpoint {ckpt} --input-dir data/test/photos \
        --output-dir outputs/phase2_test --device cuda --n 8
    print('   ✅  Comparison saved: outputs/phase1_test/ + outputs/phase2_test/')


In [ ]:
# @title 5. Save Checkpoint to Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/face2sketch'
!mkdir -p {DRIVE_DIR}/checkpoints
!mkdir -p {DRIVE_DIR}/samples

import glob, os
for ckpt in sorted(glob.glob('checkpoints/phase2_*.pt')):
    name = os.path.basename(ckpt)
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/{name}

for sample in sorted(glob.glob('samples/*.png')):
    !cp -v {sample} {DRIVE_DIR}/samples/

# Also save outputs
!mkdir -p {DRIVE_DIR}/outputs
for out in sorted(glob.glob('outputs/*.png')):
    !cp -v {out} {DRIVE_DIR}/outputs/

print('\n✅  Saved to Google Drive: MyDrive/face2sketch/')
